> ⚠️ **This notebook trains a deep-learning model and is GPU-recommended.**
> The outputs below are NOT pre-rendered — execute the notebook on a CUDA
> / MPS-equipped host to populate them. The code path was validated end
> to end in the [`omicverse#797` integration test suite][1] (`scvi-tools`
> 1.3.0, with the same kwarg-split router used here), so it is known to
> run; only the rendering depends on you executing it.
>
> Quick recipe:
> ```bash
> jupyter nbconvert --to notebook --execute --inplace \
>     docs/Tutorials-single/batch/zoo/<this-notebook>.ipynb
> ```
> on a node where `torch.cuda.is_available()` is `True`.
>
> [1]: https://github.com/omicverse/omicverse/pull/797


# Batch correction with Concord

Concord is a contrastive-learning-based integration method that uses domain (batch) labels as the negative-pair signal. Runs on GPU when available; CPU fallback exists but is slower.

This is one of the **omicverse batch-correction zoo** tutorials. See [batch/index](../index.md) for the overview / decision tree, or [../t_single_batch](../t_single_batch.ipynb) for the side-by-side comparison of every backend on a real benchmark.

## Load a 2-batch demo from pbmc3k

We use the canonical 10x pbmc3k dataset and synthesise a 2-batch label by random assignment, then plant a gene-shift on `batch_B` so the uncorrected UMAP shows a visible batch effect. This keeps the notebook self-contained and fast (~2 min end-to-end) — for a real multi-donor benchmark with [scib-metrics] scoring, see [../t_single_batch](../t_single_batch.ipynb).

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np
import pandas as pd

# Load the cached pbmc3k 10x raw counts (~2700 cells x ~32000 genes).
adata = ov.datasets.pbmc3k(processed=False)
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.layers['counts'] = adata.X.copy()  # scvi-tools needs raw counts here

# Synthesise a 2-batch demo: alternate cells, then plant a gene-shift on
# batch B so the uncorrected UMAP visibly separates by batch.
rng = np.random.default_rng(0)
batch = rng.choice(['batch_A', 'batch_B'], size=adata.n_obs, p=[0.5, 0.5])
adata.obs['batch'] = pd.Categorical(batch)

# Inject a multiplicative effect on the first 500 genes for batch_B.
from scipy.sparse import issparse, csr_matrix
X = adata.X.toarray() if issparse(adata.X) else adata.X.copy()
is_B = (adata.obs['batch'] == 'batch_B').values
X[is_B, :500] = X[is_B, :500] * 2.0
adata.X = csr_matrix(X)
adata.layers['counts'] = adata.X.copy()
adata

## Preprocess + PCA + cluster

Same QC → HVG-pearson → log-norm → PCA pipeline shared across every backend in the zoo. A quick Leiden cluster gives a synthetic `celltype` label that scANVI / scPoli can use as a prototype anchor.

In [ ]:
# Standard omicverse preprocess (QC → HVG-via-pearson → log-norm → PCA).
adata = ov.pp.qc(adata, tresh={'mito_perc': 0.2, 'nUMIs': 500,
                                 'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson', n_HVGs=2000,
                         batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features].copy()
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=30)

# Quick Leiden cluster to use as a synthetic celltype label for scANVI etc.
sc.pp.neighbors(adata, use_rep='scaled|original|X_pca', n_neighbors=15)
sc.tl.leiden(adata, resolution=0.5, flavor='igraph', directed=False,
             n_iterations=2)
adata.obs['celltype'] = adata.obs['leiden'].astype(str).map(
    lambda c: f'cluster_{c}'
).astype('category')
adata

## Uncorrected baseline

The planted batch effect is visible in the uncorrected UMAP:

In [ ]:
# Pre-correction UMAP shows the planted batch effect.
sc.tl.umap(adata, min_dist=0.3)
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
ov.pl.embedding(adata, basis='X_umap_uncorrected',
                color=['batch', 'celltype'],
                frameon='small', wspace=0.5)

## Run `ov.single.batch_correction(methods='concord')`

For the scvi-tools family backends, the wrapper auto-routes `**kwargs` between the model's `__init__` (architecture) and `.train()` (optimisation) destinations. See the **Key parameters** section below.

In [ ]:
model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='Concord',
    n_pcs=30,
)
model

## Corrected embedding

Every backend writes its corrected representation to a stable obsm key — for this one it is `adata.obsm['X_concord']`. We project via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_concord'] = ov.utils.mde(adata.obsm['X_concord'])
ov.pl.embedding(
    adata,
    basis='X_mde_concord',
    color=['batch', 'celltype'],
    frameon='small',
    wspace=0.5,
)

## Key parameters

- `domain_key` — forwarded from `batch_key`.
- `preload_dense` — set `False` for very large data to stream from disk.


## Related tutorials

- [t_batch_scvi](t_batch_scvi.ipynb) — alternative deep method.
- [t_batch_harmony](t_batch_harmony.ipynb) — CPU-friendly non-DL.

For the full side-by-side comparison with scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).